In [ ]:
!pip install -U datasets transformers accelerate bitsandbytes rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.2 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=05b68cade1adca53c7d82fcb6ac976b5a24f50fe7c302a495e5fc589497a62b4
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  

In [1]:
import json
import hashlib
import re
from datasets import load_dataset
from difflib import SequenceMatcher

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
STYLE_SYSTEM_PROMPT = "Ответь кратко, по делу, без вводных фраз и Максимум 3-4 предложения."
RAW_DATASET = "tatsu-lab/alpaca"
TARGET_SIZE = 250
MIN_LEN, MAX_LEN = 20, 800
OUTPUT_PATH = "/content/drive/MyDrive/dataset_raw_candidates.jsonl"

In [4]:
def english_target(text):
    return bool(re.search(r"[a-zA-Zа-яА-Я]", text))

In [5]:
def exact_hash(text):
    return hashlib.md5(text.strip().lower().encode()).hexdigest()

In [6]:
def near_duplicate(a, b, threshold=0.9):
    return SequenceMatcher(None, a, b).ratio() > threshold

In [7]:
ds = load_dataset(RAW_DATASET, split="train")
print(f"{len(ds)} column")

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

52002 column


In [8]:
seen_hashes = set()
kept = []
kept_texts = []

for row in ds:
    instr = row.get("instruction", "").strip()
    inp = row.get("input", "").strip()
    resp = row.get("output", "").strip()

    if inp:
        continue
    if not (MIN_LEN <= len(resp) <= MAX_LEN):
        continue

    h = exact_hash(instr + resp)
    if h in seen_hashes:
        continue
    if any(near_duplicate(resp, t) for t in kept_texts[-50:]):
        continue

    seen_hashes.add(h)
    kept_texts.append(resp)
    kept.append({"instruction": instr, "response": resp})

    if len(kept) >= TARGET_SIZE:
        break

print(f"{len(kept)}")

250


In [16]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in kept:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"saved {len(kept)} in {OUTPUT_PATH}")

saved 250 in /content/drive/MyDrive/dataset_raw_candidates.jsonl


In [19]:
!head -5 /content/drive/MyDrive/dataset_raw_candidates.jsonl

{"instruction": "Give three tips for staying healthy.", "response": "1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule."}
{"instruction": "What are the three primary colors?", "response": "The three primary colors are red, blue, and yellow."}
{"instruction": "Describe the structure of an atom.", "response": "An atom is made up of a nucleus, which contains protons and neutrons, surrounded by electrons that travel in orbits around the nucleus. The protons and neutrons have a positive charge, while the electrons have a negative charge, resulting in an overall neutral atom. The number of each particle determines the atomic number and the type of atom."}
{"instruction": "How can we reduce air pollution?", "response": "There are a number of ways to reduce air pollution, such as shifting to renewable energy sources, encouraging the use of pu